# Oscar winners for the AWARD_Movie table
Written by: Capleton

This notebook makes a clean CSV file that fits the table AWARD_Movie in our PostgreSQL database.

## STEP 1: import os and pandas
We need pandas to prepare the data before it goes into the database, and os to save the file for the import in pgAdmin.

In [1]:
import os
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## STEP 2: read the file
The original file stays unchanged. If something goes wrong in the database, we can always build the table again from the full data.

In [2]:
df = pd.read_csv(r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data\oscarAward.csv")

print("Rows and columns:", df.shape)
print(df.head())

Rows and columns: (10889, 7)
   year_film  year_ceremony  ceremony category                 name              film  winner
0       1927           1928         1    ACTOR  Richard Barthelmess         The Noose   False
1       1927           1928         1    ACTOR        Emil Jannings  The Last Command    True
2       1927           1928         1  ACTRESS       Louise Dresser   A Ship Comes In   False
3       1927           1928         1  ACTRESS         Janet Gaynor        7th Heaven    True
4       1927           1928         1  ACTRESS       Gloria Swanson    Sadie Thompson   False


## STEP 3: keep only the winners with a film
The database should only hold data that answers our research questions. An award without a film can never be linked to a row in MOVIE, so it has no place in a table with a foreign key to MOVIE.

In [3]:
winners = df[df["winner"] == True]
winners = winners.dropna(subset=["film"])

print("Rows before:", len(df))
print("Rows after:", len(winners))

Rows before: 10889
Rows after: 2172


## STEP 4: keep only the movie awards
In our ERD every type of award has its own table. AWARD_Movie holds the Oscars for the film itself. The directing and acting Oscars go to AWARD_Director and AWARD_Actor, so every Oscar is stored in exactly one table and nothing is stored twice.

In [4]:
is_director = winners["category"].str.startswith("DIRECTING")
is_actor = winners["category"].str.startswith(("ACTOR", "ACTRESS"))
award_movie = winners[~is_director & ~is_actor].copy()

print("Rows after:", len(award_movie))
print("Number of categories:", award_movie["category"].nunique())

Rows after: 1705
Number of categories: 97


## STEP 5: rename the columns
The column names in the CSV must be the same as in the table in PostgreSQL. Otherwise pgAdmin does not know which value goes into which column.

In [5]:
award_movie = award_movie.rename(columns={"name": "name_nominee", "film": "film_title"})

print(list(award_movie.columns))

['year_film', 'year_ceremony', 'ceremony', 'category', 'name_nominee', 'film_title', 'winner']


## STEP 6: clean the film titles
Some old titles in the source data contain a ;. It is a leftover and not part of the real title, so we remove it to keep the data in the database clean.

We also add film_title_clean. This is the title in small letters, a hyphen becomes a space, dots are removed and double spaces become one. The database needs this column to find the movie_id in MOVIE, because the same film can be written a bit differently in the two sources. film_title stays as it is, so the real title with capitals is still stored.

In [6]:
award_movie["film_title"] = award_movie["film_title"].str.replace(";", "").str.strip()
award_movie["film_title_clean"] = award_movie["film_title"].str.lower().str.replace("-", " ", regex=False).str.replace(".", "", regex=False).str.split().str.join(" ")

print(award_movie[["film_title", "film_title_clean"]].head())
print("Titles with ; left:", award_movie["film_title"].str.contains(";").sum())

   film_title film_title_clean
6    The Dove         the dove
9     Sunrise          sunrise
10    Sunrise          sunrise
17      Wings            wings
21      Wings            wings
Titles with ; left: 0


## STEP 7: sort the table
A fixed order gives every award the same movie_award_id every time. If the numbers change, the rows in the database no longer match the file.

In [7]:
award_movie = award_movie.sort_values(["year_ceremony", "category", "film_title"]).reset_index(drop=True)

print(award_movie[["year_ceremony", "category", "name_nominee", "film_title", "film_title_clean"]].head())

   year_ceremony             category             name_nominee film_title film_title_clean
0           1928        ART DIRECTION  William Cameron Menzies   The Dove         the dove
1           1928       CINEMATOGRAPHY           Charles Rosher    Sunrise          sunrise
2           1928       CINEMATOGRAPHY              Karl Struss    Sunrise          sunrise
3           1928  ENGINEERING EFFECTS              Roy Pomeroy      Wings            wings
4           1928  OUTSTANDING PICTURE   Paramount Famous Lasky      Wings            wings


## STEP 8: give every award a movie_award_id
movie_award_id is the primary key of AWARD_Movie. The database needs it to tell every row apart, so it must be unique and can never be empty.

In [8]:
award_movie["movie_award_id"] = award_movie.index + 1

print("First movie_award_id:", award_movie["movie_award_id"].min())
print("Last movie_award_id:", award_movie["movie_award_id"].max())
print("All unique:", award_movie["movie_award_id"].is_unique)

First movie_award_id: 1
Last movie_award_id: 1705
All unique: True


## STEP 9: add an empty movie_id
movie_id is the foreign key to MOVIE in our ERD. The column must exist to fit the table, but the number is not in the Oscar data. It is filled later with the match on title and year.

In [9]:
award_movie["movie_id"] = None

print(award_movie.head())

   year_film  year_ceremony  ceremony             category             name_nominee film_title  winner film_title_clean  movie_award_id movie_id
0       1927           1928         1        ART DIRECTION  William Cameron Menzies   The Dove    True         the dove               1     None
1       1927           1928         1       CINEMATOGRAPHY           Charles Rosher    Sunrise    True          sunrise               2     None
2       1927           1928         1       CINEMATOGRAPHY              Karl Struss    Sunrise    True          sunrise               3     None
3       1927           1928         1  ENGINEERING EFFECTS              Roy Pomeroy      Wings    True            wings               4     None
4       1927           1928         1  OUTSTANDING PICTURE   Paramount Famous Lasky      Wings    True            wings               5     None


## STEP 10: keep only the ERD columns
The CSV must have exactly the same columns, in the same order as in our ERD, as the table in PostgreSQL. winner is removed because every row is a winner, and year_film because it is always year_ceremony minus 1. Storing the same information twice goes against normalization. film_title_clean is not in the ERD. It stays in the file for now, because it is needed to fill movie_id. It is removed when movie_id is filled, before the import in pgAdmin.

In [10]:
award_movie = award_movie[["movie_id", "year_ceremony", "movie_award_id",
                              "ceremony", "category", "name_nominee", "film_title", "film_title_clean"]]

print(list(award_movie.columns))

['movie_id', 'year_ceremony', 'movie_award_id', 'ceremony', 'category', 'name_nominee', 'film_title', 'film_title_clean']


## STEP 11: save as CSV
This file is imported into AWARD_Movie in pgAdmin. index=False prevents an extra column with row numbers, because that column does not exist in the table and the import would fail.

In [12]:
folder = r"C:\Users\caple\OneDrive - HvA\Master\Data management\Data"
new_file = os.path.join(folder, "award_movie_clean.csv")

award_movie.to_csv(new_file, index=False)

print("Saved:", new_file)
print("File exists:", os.path.exists(new_file))

Saved: C:\Users\caple\OneDrive - HvA\Master\Data management\Data\award_movie_clean.csv
File exists: True
